# Leveraged ML Project » Newsletter Expert

"Goal: build a system that recaps your newsletters for you."

## Imports

In [1]:
import os
import psycopg2
import psycopg2.extras
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
load_dotenv()
NEON_PG_CONNECTION_URL = os.environ['NEON_PG_CONNECTION_URL']

ModuleNotFoundError: No module named 'psycopg2'

## DB Connection & Setup

In [ ]:
try:
    connection = psycopg2.connect(NEON_PG_CONNECTION_URL)
    connection.autocommit = True
    print("Connected to Neon Postgres!")
except Exception as e:
    print("Cannot connect to Neon Postgres:", e)

cursor = connection.cursor()

Cannot connect to Neon Postgres: name 'psycopg2' is not defined


NameError: name 'connection' is not defined

In [ ]:
try:
    cursor.execute("""CREATE EXTENSION IF NOT EXISTS vector;""")
    print("Created extension pgvector.")
except Exception as e:
    print("Cannot create extension pgvector:", e)

Created extension pgvector.


In [ ]:
try:
    cursor.execute("""DROP TABLE IF EXISTS emails;""")
    print("Dropped table emails.")
except Exception as e:
    print("Cannot drop table emails:", e)

try:
    cursor.execute(
        """
        CREATE TABLE emails (
            id BIGSERIAL PRIMARY KEY,
            part_id INT,
            title TEXT,
            body TEXT,
            vector VECTOR(384)
        );
    """
    )
    print("Created table emails.")
except Exception as e:
    print("Cannot create table emails:", e)

Dropped table emails.
Created table emails.


## Data Pre-Processing

- Re-indexing is necessary when switching models / changing model parameters
- Individual items must be indexed as they arrive

In [ ]:
# Drop existing embeddings in order to rebuild the index:
cursor.execute("""TRUNCATE emails RESTART IDENTITY;""")

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

files = os.listdir('txts')

for file in files:
    with open(f'txts/{file}', 'r') as f: text = f.read()

    # split the text into words 
    words = text.split()

    # group into chunks of less than 240 words and assign each one a part_id
    chunks = []
    chunk = []
    part_id = 0
    for word in words:
        chunk.append(word)
        if len(chunk) >= 240:
            chunks.append((part_id, ' '.join(chunk)))
            part_id += 1
            chunk = []
    if len(chunk) > 0:
        chunks.append((part_id, ' '.join(chunk)))

    # encode each chunk and save it to the database
    for part_id, chunk in chunks:
        vector = model.encode(chunk)
        cursor.execute(
            """
            INSERT INTO emails (part_id, title, body, vector)
            VALUES (%s, %s, %s, %s);
            """,
            (part_id, file, chunk, vector.tolist())  # Convert numpy array to list
        )